In [0]:
from pyspark.sql.functions import * 
from delta.tables import * 

In [0]:
%run /Workspace/Users/narutosha289@gmail.com/Consolidated_pipeline/Setup/Utilities

In [0]:
dbutils.widgets.text("Catalog","FMCG","Catalog") 
dbutils.widgets.text("Data_Source","products","Data_Source") 

Catalog = dbutils.widgets.get("Catalog")
Data_Source = dbutils.widgets.get("Data_Source")
BasePath = "s3a://sportsbar-us1/products/products.csv"
print(BasePath)

s3a://sportsbar-us1/products/products.csv


In [0]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(BasePath)
    .withColumn("read_timeStamp", current_timestamp()) 
    .select("*","_metadata.file_name","_metadata.file_size")
) 

df.show(10)

+--------------------+----------+-----------------+--------------------+------------+---------+
|        product_name|product_id|         category|      read_timeStamp|   file_name|file_size|
+--------------------+----------+-----------------+--------------------+------------+---------+
|SportsBar Energy ...|  25891101|      energy bars|2026-09-11 15:38:...|products.csv|     1388|
|SportsBar Energy ...|  25891102|      energy bars|2026-09-11 15:38:...|products.csv|     1388|
|SportsBar Energy ...|  25891103|      energy bars|2026-09-11 15:38:...|products.csv|     1388|
|SportsBar Protien...|  25891201|     protien bars|2026-09-11 15:38:...|products.csv|     1388|
|SportsBar Protien...|  25891202|     protien bars|2026-09-11 15:38:...|products.csv|     1388|
|SportsBar Protien...|  25891203|     protien bars|2026-09-11 15:38:...|products.csv|     1388|
|SportsBar Granola...|  25891301|granola & cereals|2026-09-11 15:38:...|products.csv|     1388|
|SportsBar Granola...|  25891302|granola

In [0]:
df.write.format("delta").option("enableChangeDataFeed","true").mode("overwrite").saveAsTable(f"{Catalog}.{bronze_schema}.{Data_Source}")

### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {Catalog}.{bronze_schema}.{Data_Source}") 
df_bronze.show()

+--------------------+----------+-----------------+--------------------+------------+---------+
|        product_name|product_id|         category|      read_timeStamp|   file_name|file_size|
+--------------------+----------+-----------------+--------------------+------------+---------+
|SportsBar Energy ...|  25891101|      energy bars|2026-09-11 15:39:...|products.csv|     1388|
|SportsBar Energy ...|  25891102|      energy bars|2026-09-11 15:39:...|products.csv|     1388|
|SportsBar Energy ...|  25891103|      energy bars|2026-09-11 15:39:...|products.csv|     1388|
|SportsBar Protien...|  25891201|     protien bars|2026-09-11 15:39:...|products.csv|     1388|
|SportsBar Protien...|  25891202|     protien bars|2026-09-11 15:39:...|products.csv|     1388|
|SportsBar Protien...|  25891203|     protien bars|2026-09-11 15:39:...|products.csv|     1388|
|SportsBar Granola...|  25891301|granola & cereals|2026-09-11 15:39:...|products.csv|     1388|
|SportsBar Granola...|  25891302|granola

In [0]:
# Checking for duplicates values: 
df_bronze.groupBy("product_id").count().filter(col("count") > 1).show()

+----------+-----+
|product_id|count|
+----------+-----+
|  25891101|    2|
|  25891102|    2|
+----------+-----+



In [0]:
# Drop the Duplicates: 
print("Count before dropping the duplicates:", df_bronze.count()) 
df_silver = df_bronze.dropDuplicates() 
print("Count after removing duplicates: ", df_silver.count())

Count before dropping the duplicates: 20
Count after removing duplicates:  18


In [0]:
# Fixing title case issue: 
display(df_silver.select("category").distinct())

category
energy bars
protien bars
granola & cereals
recovery dairy
healthy snacks
electrolyte mix


In [0]:
df_silver = df_silver.withColumn("category", when(df_silver["category"].isNull(), None).otherwise(initcap(df_silver["category"]))) 

In [0]:
display(df_silver.select("category").distinct())

category
Energy Bars
Protien Bars
Granola & Cereals
Recovery Dairy
Healthy Snacks
Electrolyte Mix


In [0]:
# Replace 'protien' → 'protein' in both product_name and category
df_silver = df_silver.withColumn("product_name", regexp_replace(df_silver["product_name"], "Protien", "Protein"))
df_silver = df_silver.withColumn("category", regexp_replace(df_silver["category"], "Protien", "Protein"))
df_silver.show(truncate=False)                                                            

+-------------------------------------------------+----------+-----------------+------------------------+------------+---------+
|product_name                                     |product_id|category         |read_timeStamp          |file_name   |file_size|
+-------------------------------------------------+----------+-----------------+------------------------+------------+---------+
|SportsBar Energy Bar Choco Fudge (60g)           |25891101  |Energy Bars      |2026-09-11 15:39:01.8012|products.csv|1388     |
|SportsBar Energy Bar Choco Fudge (40g)           |25891102  |Energy Bars      |2026-09-11 15:39:01.8012|products.csv|1388     |
|SportsBar Energy Bar Choco Fudge (25g)           |25891103  |Energy Bars      |2026-09-11 15:39:01.8012|products.csv|1388     |
|SportsBar Protein Bar Peanut Crunch (45g)        |25891201  |Protein Bars     |2026-09-11 15:39:01.8012|products.csv|1388     |
|SportsBar Protein Bar Peanut Crunch (55g)        |25891202  |Protein Bars     |2026-09-11 15:39:

### Standerizing Customer attributes to Match parent company data model


In [0]:
from pyspark.sql.functions import col, when, lit, regexp_extract, sha2


df_silver = (
    df_silver
    .withColumn(
        "division",
        when(col("category") == "Energy Bars", "Nutrition Bars")
        .when(col("category") == "Protein Bars", "Nutrition Bars")
        .when(col("category") == "Granola & Cereals", "Breakfast Foods")
        .when(col("category") == "Recovery Dairy", "Dairy & Recovery")
        .when(col("category") == "Healthy Snacks", "Healthy Snacks")
        .when(col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
        .otherwise("Other")
    )
)

# Generate the variant column with the quantity of the product
df_silver = df_silver.withColumn(
    "variant",
    regexp_extract(col("product_name"), r"\((.*?)\)", 1)
)

df_silver = (
    df_silver
    # 1. Generate deterministic product_code from product_name
    .withColumn(
        "product_code",
        sha2(col("product_name").cast("string"), 256)
    )
    
    # 2. Keep only numeric product_ids, otherwise use 999999
    .withColumn(
        "product_id",
        when(
            col("product_id").cast("string").rlike("^[0-9]+$"),
            col("product_id").cast("string")
        )
        .otherwise(lit(999999).cast("string"))
    )
    
    # 3. Rename product_name → product
    .withColumnRenamed("product_name", "product")
)

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")

In [0]:
display(df_silver)

product_code,division,category,product,variant,product_id,read_timestamp,file_name,file_size
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (60g),60g,25891101,2026-09-11T15:39:01.801Z,products.csv,1388
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2026-09-11T15:39:01.801Z,products.csv,1388
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (25g),25g,25891103,2026-09-11T15:39:01.801Z,products.csv,1388
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (45g),45g,25891201,2026-09-11T15:39:01.801Z,products.csv,1388
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),55g,25891202,2026-09-11T15:39:01.801Z,products.csv,1388
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (65g),65g,25891203,2026-09-11T15:39:01.801Z,products.csv,1388
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2026-09-11T15:39:01.801Z,products.csv,1388
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g,25891302,2026-09-11T15:39:01.801Z,products.csv,1388
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (200g),200g,25891303,2026-09-11T15:39:01.801Z,products.csv,1388
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (200g),200g,25891401,2026-09-11T15:39:01.801Z,products.csv,1388


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{Catalog}.{silver_schema}.{Data_Source}")

### Gold


In [0]:
df_silver = spark.sql(f"SELECT * FROM {Catalog}.{silver_schema}.{Data_Source};")
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")
df_gold.show(5)

+--------------------+----------+----------------+-----------------+--------------------+-------+
|        product_code|product_id|        division|         category|             product|variant|
+--------------------+----------+----------------+-----------------+--------------------+-------+
|fe5a8036be4b9a787...|  25891402|Dairy & Recovery|   Recovery Dairy|SportsBar Greek Y...|   120g|
|102628255d24304d6...|  25891103|  Nutrition Bars|      Energy Bars|SportsBar Energy ...|    25g|
|c68834ceaff15846b...|  25891303| Breakfast Foods|Granola & Cereals|SportsBar Granola...|   200g|
|ee1f7df9cf660ef02...|  25891501|  Healthy Snacks|   Healthy Snacks|SportsBar Oats Co...|   500g|
|5931334e4cbe6b379...|    999999|  Healthy Snacks|   Healthy Snacks|SportsBar Oats Co...|   350g|
+--------------------+----------+----------------+-----------------+--------------------+-------+
only showing top 5 rows


In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{Catalog}.{gold_schema}.sb_dim_{Data_Source}")

### Merging data source with parent source

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_products")
df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM fmcg.gold.sb_dim_products;")
df_child_products.show(5)

delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()

+--------------------+----------------+-----------------+--------------------+-------+
|        product_code|        division|         category|             product|variant|
+--------------------+----------------+-----------------+--------------------+-------+
|fe5a8036be4b9a787...|Dairy & Recovery|   Recovery Dairy|SportsBar Greek Y...|   120g|
|102628255d24304d6...|  Nutrition Bars|      Energy Bars|SportsBar Energy ...|    25g|
|c68834ceaff15846b...| Breakfast Foods|Granola & Cereals|SportsBar Granola...|   200g|
|ee1f7df9cf660ef02...|  Healthy Snacks|   Healthy Snacks|SportsBar Oats Co...|   500g|
|5931334e4cbe6b379...|  Healthy Snacks|   Healthy Snacks|SportsBar Oats Co...|   350g|
+--------------------+----------------+-----------------+--------------------+-------+
only showing top 5 rows


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]